<a href="https://colab.research.google.com/github/zegmundo-koziel/Projeto-Avaliativo-M1-S14/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Projeto de Machine Learning

**Aluno:** Zegmundo  
**Curso:** SCTEC - Machine Learning e visão computacional  
**Módulo:** M1S13  
**Data:**  07/2026

#### 🎯 Objetivo do Projeto
Desenvolver um modelo de machine learning capaz de prever se um cliente se tornará inadimplente ou se pagará seu emprestimo em dia. 

#### 🧩 Problema de Negócio
Responder qual o impacto financeiro para o banco se a máquina classificar um bom pagador como "Risco de Calote" (Falso Positivo) ou um mau pagador como "Seguro" (Falso Negativo)?

### --- Importação e configuração de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


#### Carregamento dos Dados
Nesta etapa realizamos a leitura do dataset e validação inicial.

In [ ]:
# Carregar o arquivo CSV
df = pd.read_csv('credit_risk_dataset.csv')

# Exibir as primeiras 5 linhas para verificar se os dados carregaram corretamente
print("--- Primeiras Linhas do Dataset ---")
#print(df.head())
display(df.head())

### **--- Fase 1: Análise Exploratória de Dados (EDA) ---**

Objetivo: entender a estrutura dos dados **sem realizar alterações**.

###  Analise Descritiva e Estatística

In [ ]:
# Exibir informações sobre a quantidade de colunas e linhas do dataframe
print(f"O dataframe possui {df.shape[1]} colunas e {df.shape[0]} linhas.")

# Exibir informações gerais sobre o tipo das colunas e dados nulos
print("\n--- Informações Gerais do Dataset ---")
print(df.info())

In [ ]:
# Exibir informações de valores nulos por coluna
print ("---Soma dos valores nulos por coluna---")
df.isnull().sum()

In [ ]:
# Exibir o resumo estatístico do dataframe # Média, mediana, desvio padrão e outros 
print ("---Resumo Estatístico Descritivo - Numéricas---")
display(df.describe())

print("\n---Sumário Estatístico Descritivo - Categorias---")
display(df.describe(include=['object', 'str']))

In [ ]:
# Exibir a quantidade de linhas duplicadas 
print("---Total de linhas duplicadas encontrado---")
print(f"Encontrado {df.duplicated().sum()} linhas duplicadas.")

# exibir as linhas duplicadas na tela
#print(df[df.duplicated()])


### Analise visual - gráficos

In [ ]:
# Gráfico 1: Histograma de Distribuição de Idades - person_age
plt.figure(figsize=(10, 5))
sns.histplot(df['person_age'], bins=40, kde=True, color='royalblue')
plt.title('Distribuição da Idade dos Clientes (person_age)', fontsize=14, pad=15)
plt.xlabel('Idade (Anos)', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 2: Desbalanceamento da Variável Alvo - loan_status

plt.figure(figsize=(6, 5))
ax = sns.countplot(x='loan_status', data=df, hue='loan_status', palette='Set2', legend=False)
plt.title('Desbalanceamento da Variável Alvo (loan_status)', fontsize=14, pad=15)
plt.xlabel('Status do Empréstimo (0 = Pago, 1 = Inadimplente)', fontsize=12)
plt.ylabel('Quantidade de Registros', fontsize=12)

total_registros = len(df)
for container in ax.containers:
    porcentagem = [f'{valor}\n({(valor / total_registros) * 100:.1f}%)' for valor in container.datavalues]
    ax.bar_label(container, labels=porcentagem, fontsize=10, label_type='center', color='black')

plt.tight_layout()
plt.show()


In [ ]:
# Gráfico 3: Mapa de Calor de Correlação de Pearson
plt.figure(figsize=(10, 8))
# Seleciona apenas as colunas numéricas da sua lista
colunas_numericas = df.select_dtypes(include=['int64', 'float64'])
matriz_correlacao = colunas_numericas.corr(method='pearson')

sns.heatmap(
    matriz_correlacao, 
    annot=True, 
    cmap='coolwarm', 
    fmt='.2f', 
    linewidths=0.5, 
    square=True, 
    cbar_kws={"shrink": .8},
    vmin=-1, vmax=1
)

plt.title('Mapa de Calor - Correlação de Pearson', fontsize=14, pad=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 4: Gráfico de Dispersão para Identificação de Outliers
plt.figure(figsize=(10, 6))

# Idade vs Tempo de Emprego e colorindo pelo Status do Empréstimo
sns.scatterplot(
    data=df, 
    x='person_age', 
    y='person_emp_length', 
    hue='loan_status', 
    palette='Set1', 
    alpha=0.7, 
    edgecolor='w'
)

plt.title('Gráfico de Dispersão: Idade vs. Tempo de Emprego (Identificação de Outliers)', fontsize=14, pad=15)
plt.xlabel('Idade do Cliente (person_age)', fontsize=12)
plt.ylabel('Tempo de Emprego em Anos (person_emp_length)', fontsize=12)
plt.legend(title='Status do Empréstimo\n(0 = Pago, 1 = Inadimplente)', loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


### Tomada de Decisão:

##### Tomada de Decisão e Estratégia 

A análise estatística descritiva e os quatro gráficos gerados revelaram problemas críticos que guiarão nossa estratégia de preparação na próxima fase, evitando o problema do *Garbage In, Garbage Out*:

1. **Tratamento de Outliers (Provado pelo Gráfico 1 e 4):** O gráfico de dispersão e o histograma deixam claro que existem erros gritantes de digitação na base, como pessoas com mais de 100 anos de idade (`person_age`) e tempos de emprego que ultrapassam 120 anos (`person_emp_length`). Na Fase 2, aplicaremos filtros lógicos para eliminar esses registros impossíveis (ex: manter apenas `person_age < 80` e `person_emp_length < 50`), limpando o ruído que prejudicaria os cálculos de distância do algoritmo KNN.
2. **Correção de Dados Ausentes e Duplicados:** A função `.isnull().sum()` e a checagem de duplicados acusaram a presença de dados nulos nas colunas de tempo de emprego e taxa de juros (`loan_int_rate`), além de linhas duplicadas. Removeremos as duplicadas para não gerar redundância. Como estratégia mista para dados ausentes, eliminaremos completamente as linhas com nulos em `person_emp_length` (por representar apenas 3% da base e indicar possível inconsistência cadastral) e faremos a imputação por **mediana** apenas na coluna `loan_int_rate` (10% da base), blindando nosso volume de dados contra perdas massivas e evitando distorções causadas pela assimetria.
3. **Estratégia para o Desbalanceamento (Provado pelo Gráfico 2):** O gráfico de barras confirmou que a classe majoritária (0 - Pago) representa a esmagadora maioria dos dados (aproximadamente 78%). Se ignorarmos isso, nossos modelos falharão criticamente na detecção de inadimplentes. Para resolver o problema, utilizaremos técnicas de reamostragem (como SMOTE) ou ajustaremos os pesos das classes (`class_weight='balanced'`) diretamente nos modelos KNN e Árvore de Decisão.
4. **Codificação e Escalonamento (Provado pelo Gráfico 3):** O mapa de calor de Pearson indicou baixas correlações lineares diretas com o alvo, o que justifica o uso de modelos não-lineares como as Árvores de Decisão. Para o KNN, variáveis categóricas serão transformadas via *One-Hot Encoding* e aplicaremos uma padronização rigorosa de escala (`StandardScaler`) em todas as variáveis numéricas, garantindo que o valor da renda anual (`person_income`) não soterre o peso das outras colunas no cálculo de proximidade do modelo.



### **--- Fase 2: Tratamento e Limpeza (Data Prep) ---**

### Identificação e Remoção de Duplicadas

In [ ]:
print("--- Iniciando o Pipeline de Preparação de Dados ---\n")

total_duplicadas = df.duplicated().sum()
df_tratado = df.drop_duplicates().copy()
print(f" Linhas duplicadas removidas: {total_duplicadas}")
# Exibe o tamanho do dataframe após a remoção de duplicadas
print(f"O dataframe possui {df_tratado.shape[1]} colunas e {df_tratado.shape[0]} linhas.")

### Tratamento de Valores Nulos

In [ ]:
print("\nValores nulos antes da imputação: person_emp_length")
print(df_tratado['person_emp_length'].isnull().sum())
# Remoção de Nulos da Coluna 'person_emp_length'
linhas_nulas = df_tratado.shape[0]
df_tratado = df_tratado.dropna(subset=['person_emp_length'])
linhas_nulas_removidas = linhas_nulas - df_tratado.shape[0]
print(f"Linhas nulas removidas em 'person_emp_length': {linhas_nulas_removidas} linhas")

# Imputação pela Mediana na Coluna 'loan_int_rate'
print("\nValores nulos antes da imputação: loan_int_rate")
print(df_tratado['loan_int_rate'].isnull().sum())
mediana_taxa_juros = df_tratado['loan_int_rate'].median()
print(f"Mediana da coluna 'loan_int_rate': {mediana_taxa_juros}")
df_tratado['loan_int_rate'] = df_tratado['loan_int_rate'].fillna(mediana_taxa_juros)
print(f"Valores nulos em 'loan_int_rate' após tratamento com Mediana: {df_tratado['loan_int_rate'].isnull().sum()}")
print(f"\nTamanho final do dataset limpo: {df_tratado.shape[0]} linhas e {df_tratado.shape[1]} colunas.")

### Tratamento de Outliers

In [ ]:
linhas_antes_outliers = df_tratado.shape
df_tratado = df_tratado[df_tratado['person_age'] < 80]
df_tratado = df_tratado[df_tratado['person_emp_length'] < 50] 
linhas_removidas_outliers = linhas_antes_outliers[0] - df_tratado.shape[0]
print(f"Outliers extremos removidos (Idade > 80 ou Emprego > 50): {linhas_removidas_outliers} linhas")


In [ ]:
# diferença de linhas e colunas entre o dataset original e o dataset tratado
total_linhas_removidas = df.shape[0] - df_tratado.shape[0]
porcentagem_linhas_removidas = (total_linhas_removidas / df.shape[0]) * 100
# Linhas tratadas com a mediana
linhas_tratadas_mediana = df['loan_int_rate'].isnull().sum()
porcentagem_linhas_tratadas = (linhas_tratadas_mediana / df.shape[0]) * 100

print("--- RESUMO DO TRATAMENTO DE DADOS ---")
print(f"- Total de linhas removidas durante o tratamento: {total_linhas_removidas} linhas")   
print(f"- Porcentagem de linhas removidas: {porcentagem_linhas_removidas:.2f}%")   
print(f"- Linhas tratadas com mediana: {linhas_tratadas_mediana} linhas") 
print(f"- Porcentagem de linhas tratadas com mediana: {porcentagem_linhas_tratadas:.2f}%")


In [ ]:
# Gráfico 5: Gráfico de Dispersão de Outliers pós-tratamento
plt.figure(figsize=(10, 6))

# Idade vs Tempo de Emprego e colorindo pelo Status do Empréstimo
sns.scatterplot(
    data=df_tratado, 
    x='person_age', 
    y='person_emp_length', 
    hue='loan_status', 
    palette='Set1', 
    alpha=0.7, 
    edgecolor='w'
)

plt.title('Gráfico de Dispersão: Idade vs. Tempo de Emprego (Pós-Tratamento de Outliers)', fontsize=14, pad=15)
plt.xlabel('Idade do Cliente (person_age)', fontsize=12)
plt.ylabel('Tempo de Emprego em Anos (person_emp_length)', fontsize=12)
plt.legend(title='Status do Empréstimo\n(0 = Pago, 1 = Inadimplente)', loc='upper right')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 6: Boxplots para Validação de Outliers Pós-Tratamento
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Boxplot 1: Idade do Cliente por Status do Empréstimo
sns.boxplot(
    data=df_tratado, 
    x='loan_status', 
    y='person_age', 
    ax=axes[0],               
    hue='loan_status',        
    palette='Set2', 
    legend=False              
)
axes[0].set_title('Distribuição da Idade (person_age)', fontsize=12, pad=10)
axes[0].set_xlabel('Status do Empréstimo (0=Pago, 1=Inadimplente)', fontsize=10)
axes[0].set_ylabel('Idade em Anos', fontsize=10)
axes[0].grid(True, linestyle='--', alpha=0.3)

# Boxplot 2: Tempo de Emprego por Status do Empréstimo
sns.boxplot(
    data=df_tratado, 
    x='loan_status', 
    y='person_emp_length', 
    ax=axes[1],               
    hue='loan_status',        
    palette='Set2', 
    legend=False              
)
axes[1].set_title('Tempo de Emprego (person_emp_length)', fontsize=12, pad=10)
axes[1].set_xlabel('Status do Empréstimo (0=Pago, 1=Inadimplente)', fontsize=10)
axes[1].set_ylabel('Anos de Emprego', fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.3)

plt.suptitle('Boxplots de Controle Pós-Tratamento de Outliers', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

##### Justificativa Técnica para tratamento

**Justificativa para a Abordagem Combinada de Dados Ausentes:**  
Nesta etapa, optou-se por uma estratégia mista para o tratamento de dados nulos, avaliando o impacto de cada variável tanto do ponto de vista estatístico quanto do negócio:
1. **Remoção em `person_emp_length` (Tempo de Emprego):** O tempo de emprego é um indicador crucial para a análise de risco de crédito. A ausência desse dado pode sugerir inconsistência cadastral ou ocultação de informação (como desemprego). Como o volume de dados nulos nesta coluna representava um percentual muito baixo (aproximadamente 3% da base), a remoção completa dessas linhas limpa o dataset sem causar perda significativa de informação ou poder preditivo.
2. **Imputação por Mediana em `loan_int_rate` (Taxa de Juros):** Diferente da anterior, a taxa de juros possuía quase 10% de dados ausentes. Eliminar todas essas linhas causaria um descarte massivo de dados úteis (*data wastage*), reduzindo drasticamente a capacidade de aprendizado dos modelos. Optou-se pela **Mediana** por ser uma métrica de tendência central robusta, ideal para distribuições com forte assimetria e presença de *outliers* (como visto na EDA). A média distorceria o valor imputado para cima ou para baixo devido às taxas extremas, enquanto a mediana garante neutralidade estatística.

**Decisão de Tratamento de Outliers e Impacto nos Modelos:**  
Registros com dados impossíveis ou severamente discrepantes (idade superior ou igual a 80 anos e tempo de emprego incompatível com uma carreira profissional ativa) foram identificados nos gráficos da EDA e cirurgicamente **removidos**, garantindo que a massa de dados final represente clientes reais do mercado de crédito atual.

O impacto dessa higienização reflete diretamente na arquitetura dos algoritmos que serão testados:
* **Algoritmo KNN (K-Nearest Neighbors):** É **altamente vulnerável** a outliers. Como o KNN calcula a proximidade entre clientes usando fórmulas de distância geométrica no espaço multidimensional, a presença de valores absurdos de idade ou tempo de emprego distorceria a escala espacial inteira, aproximando incorretamente perfis totalmente diferentes.
* **Árvore de Decisão:** É **inerentemente robusta** a outliers, pois faz divisões por regras condicionais simples (ex: `taxa_juros > 12%`). O algoritmo não se importa com a magnitude do extremo, apenas com o ponto de quebra. Contudo, restringir a idade para menores de 80 anos e ajustar o tempo de emprego evita a criação de ramificações inúteis e folhas isoladas para comportamentos bizarros, diminuindo as chances de *overfitting* (sobreajuste).


### **---Fase 3: Feature Engineering (Coluna Calculada)---**

In [ ]:
# Criando a nova coluna de comprometimento de renda
df_tratado["comprometimento_renda"] = (
    df_tratado["loan_amnt"] / df_tratado["person_income"]
) * 100

# Visualizando as primeiras linhas apenas das colunas utilizadas para o cálculo do comprometimento de renda
df_tratado[["loan_amnt", "person_income", "comprometimento_renda"]].head()

In [ ]:
# Resumo descritivo da nova coluna
print("--- Estatísticas do Comprometimento de Renda ---")
print(df_tratado["comprometimento_renda"].describe())

# Gráfico 7: Boxplot para identificar outliers visualmente
plt.figure(figsize=(8, 4))
sns.boxplot(x=df_tratado["comprometimento_renda"], color="skyblue")
plt.title("Distribuição do Comprometimento de Renda")
plt.xlabel("Comprometimento de Renda (%)")
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.show()

### **--- Fase 4: Separação, Balanceamento e Escalonamento Seguro ---**

### Split de dados

In [ ]:
# Separando variáveis preditoras (X) do alvo (y)
X = df_tratado.drop(columns=['loan_status'])
y = df_tratado['loan_status']

 # Divisão em Treino (80%) e Teste (20%) com estratificação para manter a proporção de inadimplentes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Identificando os tipos de colunas nos dados de treino
colunas_cat = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()
colunas_num = X_train.select_dtypes(exclude=["object", "category", "string"]).columns.tolist()


### Encoding

In [ ]:
# Instancia o codificador para remover a primeira coluna (drop='first')
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='error')

# Ajusta e transforma o treino / Apenas transforma o teste
encoded_train = ohe.fit_transform(X_train[colunas_cat])
encoded_test = ohe.transform(X_test[colunas_cat])

# Recupera os nomes das novas colunas geradas
nomes_novas_colunas = ohe.get_feature_names_out(colunas_cat)

# Reconstrói os DataFrames de treino e teste unindo numéricas e categóricas codificadas
X_train_encoded = pd.concat([
    X_train[colunas_num].reset_index(drop=True),
    pd.DataFrame(encoded_train, columns=nomes_novas_colunas)
], axis=1)

X_test_encoded = pd.concat([
    X_test[colunas_num].reset_index(drop=True),
    pd.DataFrame(encoded_test, columns=nomes_novas_colunas)
], axis=1)

### Escalonamento Seguro

In [ ]:
# Escalonamento Seguro (Apenas variáveis contínuas)
continuous_cols = ['person_age', 'person_income', 'person_emp_length', 
                   'loan_amnt', 'loan_int_rate', 'loan_percent_income', 
                   'cb_person_cred_hist_length', 'comprometimento_renda']

scaler = StandardScaler()

# Criamos as bases para o KNN
X_train_knn_pre_bal = X_train_encoded.copy()
X_test_knn = X_test_encoded.copy()

# Fit_transform apenas no treino, transform no teste (Evita Data Leakage)
X_train_knn_pre_bal[continuous_cols] = scaler.fit_transform(X_train_encoded[continuous_cols])
X_test_knn[continuous_cols] = scaler.transform(X_test_encoded[continuous_cols])

### Balanceamento de Classes

In [ ]:
# Aplicando SMOTE estritamente e APENAS nos dados de treino
smote = SMOTE(random_state=42)

# Geramos os dados balanceados e escalonados para o KNN
X_train_knn, y_train_bal = smote.fit_resample(X_train_knn_pre_bal, y_train)

# Geramos os dados balanceados puros  para a Árvore de Decisão
X_train_bal, _ = smote.fit_resample(X_train_encoded, y_train)

##### Para a Árvore de Decisão, 
usamos as variáveis brutas (X_train e X_test) sem escalonamento, pois algoritmos de árvore tomam decisões baseadas em limiares locais, sendo puramente independentes da escala global das variáveis.

### **--- Fase 5: Modelagem e Validação (O Desafio do Overfitting) ---**

### Otimização do KNN

In [ ]:
print(" --- DIAGNÓSTICO DE OVERFITTING ---")
k_valores = [3, 5, 7, 9, 11]
knn_resultados = []

for k in k_valores:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_knn, y_train_bal) 
    acc_train = accuracy_score(y_train_bal, knn.predict(X_train_knn)) 
    acc_test = accuracy_score(y_test, knn.predict(X_test_knn))
    knn_resultados.append({'K': k, 'Treino': acc_train, 'Teste': acc_test})

df_knn_perf = pd.DataFrame(knn_resultados)
print("\nPerformance KNN:")
print(df_knn_perf.to_string(index=False))

### Otimização da Árvore de Decisão

In [ ]:
profundidades = [3, 5, 7, 9, 11, 13, 15, None]
tree_resultados = []

for d in profundidades:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train_bal, y_train_bal)
    acc_train = accuracy_score(y_train_bal, tree.predict(X_train_bal))
    acc_test = accuracy_score(y_test, tree.predict(X_test_encoded)) 
    tree_resultados.append({'Max_Depth': str(d), 'Treino': acc_train, 'Teste': acc_test})

df_tree_perf = pd.DataFrame(tree_resultados)
print("\nPerformance Árvore de Decisão:")
print(df_tree_perf.to_string(index=False))

# Seleção dos melhores modelos com base nos testes
melhor_knn = KNeighborsClassifier(n_neighbors=9).fit(X_train_knn, y_train_bal)
melhor_tree = DecisionTreeClassifier(max_depth=11, random_state=42).fit(X_train_bal, y_train_bal)


##### EXPLICAÇÃO DO OVERFITTING:
Na Árvore de Decisão com 'max_depth=None' (ilimitada), o modelo atingiu quase 100% no treino 
e caiu drasticamente no teste, provando que ele apenas "decorou" os dados (overfitting). 
A configuração que evitou o overfitting e garantiu a melhor capacidade de generalização 
foi 'max_depth=11' para a Árvore de Decisão, mantendo métricas equilibradas entre treino e teste. 
Para o KNN, o valor de K=9 apresentou o melhor equilíbrio de estabilidade.

### **--- Fase 6: Avaliação e veredito de Negócios ---**


In [ ]:
print("\n=== MATRIZES DE CONFUSÃO E RELATÓRIOS ===")


y_pred_knn = melhor_knn.predict(X_test_knn)
y_pred_tree = melhor_tree.predict(X_test_encoded) 

# Relatório do KNN
print("\n--- [Relatório de Classificação - KNN (K=9)] ---")
print(classification_report(y_test, y_pred_knn))

# Relatório da Árvore
print("\n--- [Relatório de Classificação - Árvore (Profundidade=11)] ---")
print(classification_report(y_test, y_pred_tree))


In [ ]:
# # Gráfico 8: figura com dois subplots lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de Confusão - KNN
cm_knn = confusion_matrix(y_test, y_pred_knn)
disp_knn = ConfusionMatrixDisplay(confusion_matrix=cm_knn, display_labels=['Bom Pagador (0)', 'Inadimplente (1)'])
disp_knn.plot(ax=axes[0], cmap='Blues', values_format='d') 
axes[0].set_title('Matriz de Confusão - KNN (K=9)', fontsize=12, pad=10)

# Matriz de Confusão - Árvore de Decisão
cm_tree = confusion_matrix(y_test, y_pred_tree)
disp_tree = ConfusionMatrixDisplay(confusion_matrix=cm_tree, display_labels=['Bom Pagador (0)', 'Inadimplente (1)'])
disp_tree.plot(ax=axes[1], cmap='Greens', values_format='d')
axes[1].set_title('Matriz de Confusão - Árvore (Profundidade=11)', fontsize=12, pad=10)

plt.suptitle('Comparativo Visual das Matrizes de Confusão', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()


#####  Análise de Negócio e Veredito Final
No cenário de concessão e risco de crédito, a avaliação estatística precisa ser traduzida diretamente em termos de impacto financeiro e custo de oportunidade. Para isso, devemos analisar os dois tipos de erros cometidos pelos modelos:

1. **Falso Positivo (O modelo prevê que o cliente é Inadimplente, mas ele pagaria em dia):**
   * **Impacto:** O banco recusa o crédito para um cliente saudável. Gera um **custo de oportunidade** e perda de receita potencial (juros que deixaram de ser ganhos), além de uma possível insatisfação do cliente que migrará para o concorrente.
2. **Falso Negativo (O modelo prevê que o cliente é um Bom Pagador, mas ele se torna Inadimplente):**
   * **Impacto:** O banco aprova o empréstimo para alguém que não vai pagar. Gera **prejuízo financeiro direto e imediato** (*default*), onde o banco perde o valor principal emprestado (capital de giro).

#####  Qual erro gera o pior impacto financeiro?
Em finanças, **o Falso Negativo é severamente pior do que o Falso Positivo**. Perder o dinheiro principal emprestado destrói o caixa de uma instituição muito mais rápido do que deixar de lucrar juros sobre uma venda perdida. Portanto, o banco precisa de um modelo que minimize os Falsos Negócios (buscando um bom *Recall* na classe 1), mas que mantenha uma eficiência operacional mínima (boa *Precision* na classe 1) para não travar a operação comercial.

#####  Veredito Final: Modelo Escolhido para Produção
O modelo selecionado para entrar em produção é a **Árvore de Decisão (Profundidade=7)**.

**Justificativa Técnica e Financeira:**
* **Eficiência Geral:** A Árvore de Decisão obteve uma acurácia global de **91%**, contra 84% do KNN. Ela erra muito menos no volume total da carteira.
* **Proteção Operacional e Comercial (Precision de 87% vs 61%):** O KNN apresentou uma precisão de apenas 61% na detecção de inadimplentes. Isso significa que a cada 100 bloqueios que o KNN faz, 39 seriam alarmes falsos em cima de bons clientes. A Árvore de Decisão eleva essa precisão para 87% (apenas 13% de alarmes falsos). Isso protege o faturamento comercial do banco, evitando o bloqueio em massa de clientes legítimos.
* **Equilíbrio Financeiro (F1-Score de 0.77 vs 0.68):** Embora o KNN tenha capturado uma fatia ligeiramente maior de inadimplentes isolados (*Recall* de 76% vs 69%), o custo operacional de carregar 39% de alarmes falsos inviabilizaria o negócio do banco. A Árvore de Decisão entrega o melhor equilíbrio (*F1-Score* substancialmente maior), garantindo segurança ao crédito sem sufocar o crescimento da carteira de clientes.
